In [ ]:
# flake8: noqa: T201

import os

import torch
from lightning.pytorch import seed_everything
from mattergen.common.data.datamodule import CrystDataModule
from mattergen.diffusion.corruption.multi_corruption import MultiCorruption
from mattergen.diffusion.diffusion_module import DiffusionModule
from mattergen.diffusion.losses import DenoisingScoreMatchingLoss, Loss
from mattergen.diffusion.model_target import ModelTarget
from mattergen.diffusion.score_models.base import ScoreModel  # noqa: TC002
from mattergen.diffusion.timestep_samplers import UniformTimestepSampler
from omegaconf import OmegaConf

from src.kldm.data.dataset import MP20
from src.kldm.data.transform import ConcatFeatures, ContinuousIntervalLattice, FullyConnectedGraph, UnsqueezeLattice
from src.kldm.diffusion.score_models.score_model import TinyScoreModel
from src.kldm.diffusion.sde import VPSDE
from src.kldm.diffusion.wrapped_sde import WrappedVESDE

os.chdir("/workspace")

/workspace/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.12/site-packages/mattergen


In [3]:
seed_everything(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

Seed set to 42


## Data

### Dataset

Input: list of `Transforms`, root location

Output: `CrystalDataset`, accessible with `.data`

In [ ]:
transforms = [
    FullyConnectedGraph(),  # fully connected graph representation of the crystal structure
    ContinuousIntervalLattice(),  # continuous interval representation of lattice parameters
    ConcatFeatures(in_keys=["lengths", "angles"], out_key="l"),  # concatenation of lengths and angles into a single feature vector "l"
    UnsqueezeLattice(),  # add an extra dimension to the lattice parameters for compatibility with model input
    # CellToLattice(),  # convert cell parameters to lattice parameters for compatibility with model input  # noqa: ERA001
]

In [5]:
train_dataset = MP20(root="data", split="train", download=True, transforms=transforms)  # ty:ignore[invalid-argument-type]
test_dataset = MP20(root="data", split="test", download=True, transforms=transforms)  # ty:ignore[invalid-argument-type]
val_dataset = MP20(root="data", split="val", download=True, transforms=transforms)  # ty:ignore[invalid-argument-type]

### Data Module

Input: `CrystalDataset` (or wrapper) and batch size configs etc.

Output: `CrystDataModule` which can generate batches `ChemGraphBatch`

In [ ]:
batch_size = OmegaConf.create(
    {
        "train": 4,
        "val": 4,
        "test": 4,
    }
)

num_workers = OmegaConf.create(
    {
        "train": 0,
        "val": 0,
        "test": 0,
    }
)

data_module = CrystDataModule(
    train_dataset=train_dataset,  # ty:ignore[invalid-argument-type]
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
)

### Data Loader

In [7]:
loader = data_module.train_dataloader()

batch = next(iter(loader))

print(batch.cell)

tensor([[[ 3.5200e-01, -2.8810e+00, -0.0000e+00],
         [-0.0000e+00, -0.0000e+00,  5.0456e+00],
         [-1.1965e+01, -0.0000e+00, -1.0084e+00]],

        [[-4.8718e+00,  0.0000e+00, -1.7190e+00],
         [-2.4359e+00, -7.3206e+00, -8.5951e-01],
         [-4.8718e+00,  0.0000e+00,  6.0440e+00]],

        [[-1.5458e+00, -2.6773e+00, -3.7860e-16],
         [-1.5458e+00,  2.6773e+00,  1.8930e-16],
         [ 0.0000e+00,  0.0000e+00, -7.2440e+00]],

        [[ 8.8518e-16,  5.5044e+00,  2.8122e-03],
         [-0.0000e+00, -0.0000e+00,  5.6059e+00],
         [ 7.8444e+00, -0.0000e+00,  4.8033e-16]]])


## Diffusion

### Score Model

Input: `forward` method needs `x` which is a `ChemGraph` (noisy batch) and `t` which is a timestep

Output: tries to approximate noise $-\epsilon$ for each field

In [8]:
score_model: ScoreModel = TinyScoreModel(
    hidden_dim=64,
)

score_model: ScoreModel = CSPNetDenoiser()

### Corruption

Input: `SDE`s for various fields. In this case, we have:
- one for diffusion of fractional coordinates (`pos`)
- one for diffusion of cell/lattice (`cell`)

Output: `MultiCorruption` which allows for sampling marginals via `sample_marginal`

Basically tells what kind of corruption (in this case SDE) that should corrupt each of the $k$ fields.

In [9]:
multi_corruption = MultiCorruption(
    sdes={
        "pos": WrappedVESDE(),  # for diffusion on fractional coordinates
        "cell": VPSDE(),  # for diffusion on lattice parameters
    }
)

multi_corruption = MultiCorruption(
    sdes={
        "pos": NumAtomsVarianceAdjustedWrappedVESDE(wrapping_boundary=1.0),  # for diffusion on fractional coordinates
        "cell": LatticeVPSDE(),  # for diffusion on lattice parameters
    }
)

### Loss Function

Input: `ModelTargets`

Output: mean of aggregated loss across fields

In [10]:
model_targets = {
    "pos": ModelTarget.score_times_std,
    "cell": ModelTarget.score_times_std,
}

loss_fn: Loss = DenoisingScoreMatchingLoss(
    model_targets=model_targets,
    reduce="mean",
    weights={"pos": 1.0, "cell": 1.0},  # {"pos": 0.1, "cell": 1.0},
)

loss_fn: Loss = MaterialsLoss(
    reduce="sum",
    include_pos=True,
    include_cell=True,
    include_atomic_numbers=False,
    weights={
        "cell": 1.0,
        "pos": 0.1,
    },
)

### Timestep Sampling

Input: $t_{\min}$ and $t_{\max}$ of uniform distribution

Output: $\bm{t} \in [t_{\min}, t_{\max}]^{B}$ where $B$ is batch size, and $\bm{t}$ is a tensor of uniformly random samples

In [11]:
time_sampler = UniformTimestepSampler(min_t=1e-5, max_t=1)

time_sampler(batch_size=batch.batch_size, device=batch.pos.device)

tensor([0.9199, 0.1247, 0.3573, 0.6168])

### Diffusion Module

In [12]:
diffusion_module = DiffusionModule(
    model=score_model,
    corruption=multi_corruption,
    loss_fn=loss_fn,
    pre_corruption_fn=None,
    timestep_sampler=time_sampler,
)

In [ ]:
batch = next(iter(loader)).to(device)

num_steps = 2000

score_model.train()

with torch.no_grad():
    noisy_batch, t = diffusion_module._corrupt_batch(batch)  # noqa: SLF001

In [14]:
optimizer = torch.optim.Adam(score_model.parameters(), lr=1e-4)

loss_hist = []

for step in range(num_steps):
    optimizer.zero_grad()

    pred = score_model(noisy_batch, t)

    loss, metrics = loss_fn(
        multi_corruption=multi_corruption,
        batch=batch,
        noisy_batch=noisy_batch,
        score_model_output=pred,
        t=t,
    )

    loss.backward()
    optimizer.step()

    loss_hist.append(loss.item())

    if step % 100 == 0:
        print(f"step {step:4d} | loss={loss.item():.4f} | pos={metrics['pos'].item():.4f} | cell={metrics['cell'].item():.4f}")

step    0 | loss=1.6559 | pos=0.5397 | cell=1.1162
step  100 | loss=1.3398 | pos=0.5056 | cell=0.8341
step  200 | loss=1.0852 | pos=0.4885 | cell=0.5967
step  300 | loss=0.8651 | pos=0.4766 | cell=0.3885
step  400 | loss=0.6888 | pos=0.4636 | cell=0.2252
step  500 | loss=0.5608 | pos=0.4482 | cell=0.1125
step  600 | loss=0.4752 | pos=0.4298 | cell=0.0454
step  700 | loss=0.4230 | pos=0.4090 | cell=0.0140
step  800 | loss=0.3911 | pos=0.3879 | cell=0.0032
step  900 | loss=0.3705 | pos=0.3699 | cell=0.0005
step 1000 | loss=0.3579 | pos=0.3579 | cell=0.0001
step 1100 | loss=0.3515 | pos=0.3515 | cell=0.0000
step 1200 | loss=0.3483 | pos=0.3483 | cell=0.0000
step 1300 | loss=0.3463 | pos=0.3463 | cell=0.0000
step 1400 | loss=0.3447 | pos=0.3447 | cell=0.0000
step 1500 | loss=0.3432 | pos=0.3432 | cell=0.0000
step 1600 | loss=0.3414 | pos=0.3414 | cell=0.0000
step 1700 | loss=0.3394 | pos=0.3394 | cell=0.0000
step 1800 | loss=0.3369 | pos=0.3369 | cell=0.0000
step 1900 | loss=0.3338 | pos=0

## Sanity Checks

### Fixed overfitting on a single batch

In [15]:
# Pick a single batch
batch = next(iter(loader)).to(batch.pos.device)

score_model.train()
optimizer = torch.optim.Adam(score_model.parameters(), lr=1e-4)

loss_hist = []

for step in range(2000):  # fewer steps for overfit test
    optimizer.zero_grad()

    # Add noise deterministically by fixing the timestep
    t = torch.full((batch.batch_size,), 0.5, device=batch.pos.device)
    noisy_batch = multi_corruption.sample_marginal(batch, t)

    pred = score_model(noisy_batch, t)
    loss, metrics = loss_fn(
        multi_corruption=multi_corruption,
        batch=batch,
        noisy_batch=noisy_batch,
        score_model_output=pred,
        t=t,
    )

    loss.backward()
    optimizer.step()
    loss_hist.append(loss.item())

    if step % 50 == 0:
        print(f"step {step:4d} | loss={loss.item():.4f} | pos={metrics['pos'].item():.4f} | cell={metrics['cell'].item():.4f}")

step    0 | loss=1.5020 | pos=0.2979 | cell=1.2041
step   50 | loss=1.4894 | pos=0.1944 | cell=1.2949
step  100 | loss=1.3859 | pos=0.2241 | cell=1.1618
step  150 | loss=0.9635 | pos=0.1834 | cell=0.7801
step  200 | loss=0.7925 | pos=0.2111 | cell=0.5814
step  250 | loss=1.1443 | pos=0.1673 | cell=0.9770
step  300 | loss=0.9471 | pos=0.1748 | cell=0.7724
step  350 | loss=0.8167 | pos=0.1799 | cell=0.6368
step  400 | loss=0.9891 | pos=0.2022 | cell=0.7869
step  450 | loss=0.7507 | pos=0.1774 | cell=0.5732
step  500 | loss=0.8448 | pos=0.1756 | cell=0.6692
step  550 | loss=0.6740 | pos=0.1707 | cell=0.5033
step  600 | loss=0.7885 | pos=0.1623 | cell=0.6262
step  650 | loss=0.6596 | pos=0.1593 | cell=0.5003
step  700 | loss=0.7348 | pos=0.1772 | cell=0.5576
step  750 | loss=0.8590 | pos=0.1662 | cell=0.6928
step  800 | loss=0.5921 | pos=0.1690 | cell=0.4232
step  850 | loss=0.8252 | pos=0.1656 | cell=0.6596
step  900 | loss=0.7775 | pos=0.1664 | cell=0.6111
step  950 | loss=0.5840 | pos=0

## Training Loop

In [ ]:
batch_size = OmegaConf.create(
    {
        "train": 32,
        "val": 32,
        "test": 32,
    }
)

num_workers = OmegaConf.create(
    {
        "train": 0,
        "val": 0,
        "test": 0,
    }
)

data_module = CrystDataModule(
    train_dataset=train_dataset,  # ty:ignore[invalid-argument-type]
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
)

In [17]:
train_loader = data_module.train_dataloader()
test_loader = data_module.test_dataloader()
val_loader = data_module.val_dataloader()

In [18]:
val_loss_ls = []
train_loss_ls = []

In [ ]:
num_epochs = 50

train_loss = 0.0

for epoch in range(num_epochs):
    score_model.train()
    for batch in train_loader:
        batch = batch.to(device)  # noqa: PLW2901
        optimizer.zero_grad()
        loss, metrics = diffusion_module.calc_loss(batch)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()
    train_loss /= len(train_loader)

    # Validation
    score_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_batch in val_loader:
            val_batch = val_batch.to(device)  # noqa: PLW2901
            l, _ = diffusion_module.calc_loss(val_batch)  # noqa: E741
            val_loss += l.item()
    val_loss /= len(val_loader)
    train_loss_ls.append(train_loss)
    val_loss_ls.append(val_loss)
    print(f"Epoch {epoch + 1}: val_loss={val_loss:.4f}, train_loss={train_loss:.4f}")

Epoch 0: val_loss=49.1363, train_loss=50.0339
Epoch 1: val_loss=48.8019, train_loss=49.1854
Epoch 2: val_loss=46.1065, train_loss=48.2836
Epoch 3: val_loss=46.7026, train_loss=45.2002
Epoch 4: val_loss=43.2951, train_loss=44.4223
Epoch 5: val_loss=45.1094, train_loss=46.4966
Epoch 6: val_loss=42.5555, train_loss=43.0392
Epoch 7: val_loss=44.9582, train_loss=45.3413
Epoch 8: val_loss=41.0172, train_loss=43.2964
Epoch 9: val_loss=41.2434, train_loss=41.0381
Epoch 10: val_loss=40.3847, train_loss=42.5898
Epoch 11: val_loss=41.3255, train_loss=41.8592
Epoch 12: val_loss=42.0398, train_loss=41.3182
Epoch 13: val_loss=42.0275, train_loss=40.8135
Epoch 14: val_loss=40.5327, train_loss=41.5720
Epoch 15: val_loss=41.2422, train_loss=39.6490
Epoch 16: val_loss=42.5845, train_loss=40.4794
Epoch 17: val_loss=38.9274, train_loss=39.7792
Epoch 18: val_loss=40.5082, train_loss=39.8923
Epoch 19: val_loss=38.2265, train_loss=36.2610
Epoch 20: val_loss=37.5907, train_loss=38.8250
Epoch 21: val_loss=39.2